# Taylor Monetary-Policy Shock

This notebook trains separate rule-based DEQN networks for a one-time monetary-policy shock.  The shock is an additional state, \(\varepsilon_t^R\), and the Taylor rule is implemented as \(R_t=R_t^{rule}\exp(\varepsilon_t^R)\).  The default set covers fixed Taylor, natural-rate-adjusted Taylor, cap-pressure bottleneck Taylor, and repair-aware Taylor policies.

In [ ]:
# Set repository paths and make the local package importable.
from pathlib import Path
import os
import subprocess
import sys
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC) + os.pathsep + env.get("PYTHONPATH", "")

ARTIFACT_ROOT = ROOT / "baseline_artifacts" / "critical_input_deqn"
SHOCK_ROOT = ARTIFACT_ROOT / "rule_monetary_shock"
NATURAL_CANDIDATES = [
    ARTIFACT_ROOT / "natural" / "checkpoints" / "natural_best.pt",
    ARTIFACT_ROOT / "natural" / "natural.pt",
    ARTIFACT_ROOT / "natural.pt",
]
NATURAL_CHECKPOINT = next((path for path in NATURAL_CANDIDATES if path.exists()), NATURAL_CANDIDATES[0])

ROOT, SRC, SHOCK_ROOT, NATURAL_CHECKPOINT

# Stream subprocess output line by line in Colab instead of waiting silently.
def run_stream(cmd, *, cwd=ROOT, env=None):
    print('Running:', ' '.join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, cmd)



In [ ]:
# Configure the Taylor monetary-shock training run.
POLICIES = "fixed,ba,bottleneck,repair_aware"
RULE_STEPS = 8_000
LOG_EVERY = 100
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float64"

# The shock is measured as annualized basis points and converted inside the postprocess script.
SMALL_BP_ANNUALIZED = 25
LARGE_BP_ANNUALIZED = 100
RHO_R_SHOCK = 0.50

# Keep the same stopping logic as the baseline Taylor notebooks.
BATCH_SIZE = 2048
SIM_BATCH_SIZE = 512
EPISODE_LENGTH = 20
EPISODE_UPDATES_PER_EPISODE = 2
EPISODE_BROAD_SHARE = 0.50
CHECKPOINT_EVERY = 1000
SCENARIO_Q_WEIGHT = 25.0
CALM_ANCHOR_WEIGHT = 5.0
SCENARIO_BURNIN = 5
SCENARIO_HORIZON = 10
SCENARIO_LOSS_INTERVAL = 25
QMC_TRAIN = 256
QMC_VAL = 512


In [ ]:
# Train fixed, natural-rate-adjusted, cap-pressure bottleneck, and repair-aware Taylor with eps_R as an extra state.
cmd = [
    sys.executable,
    "-u",
    "-m",
    "critical_input_deqn.run_rule_monetary_shock",
    "--output-dir", str(SHOCK_ROOT),
    "--natural-checkpoint", str(NATURAL_CHECKPOINT),
    "--policies", POLICIES,
    "--rule-steps", str(RULE_STEPS),
    "--log-every", str(LOG_EVERY),
    "--episode-updates-per-episode", str(EPISODE_UPDATES_PER_EPISODE),
    "--episode-broad-share", str(EPISODE_BROAD_SHARE),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--rule-scenario-q-weight", str(SCENARIO_Q_WEIGHT),
    "--rule-calm-anchor-weight", str(CALM_ANCHOR_WEIGHT),
    "--rule-scenario-burnin", str(SCENARIO_BURNIN),
    "--rule-scenario-horizon", str(SCENARIO_HORIZON),
    "--rule-scenario-loss-interval", str(SCENARIO_LOSS_INTERVAL),
    "--device", DEVICE,
    "--dtype", DTYPE,
    "--rho-R-shock", str(RHO_R_SHOCK),
    "--small-bp-annualized", str(SMALL_BP_ANNUALIZED),
    "--large-bp-annualized", str(LARGE_BP_ANNUALIZED),
    "--batch-size", str(BATCH_SIZE),
    "--sim-batch-size", str(SIM_BATCH_SIZE),
    "--episode-length", str(EPISODE_LENGTH),
    "--qmc-train", str(QMC_TRAIN),
    "--qmc-val", str(QMC_VAL),
]
print(" ".join(cmd))
run_stream(cmd, cwd=ROOT, env=env)


In [ ]:
# Generate deterministic IRFs for one-time monetary shocks under the two Taylor rules.
cmd = [
    sys.executable,
    "-m",
    "critical_input_deqn.postprocess_rule_monetary_shock",
    "--artifact-root", str(SHOCK_ROOT),
    "--output-dir", str(SHOCK_ROOT / "postprocess"),
    "--natural-checkpoint", str(NATURAL_CHECKPOINT),
    "--policies", POLICIES,
    "--ir-burnin", "200",
    "--ir-horizon", "80",
    "--ir-presteps", "5",
    "--rho-R-shock", str(RHO_R_SHOCK),
    "--small-bp-annualized", str(SMALL_BP_ANNUALIZED),
    "--large-bp-annualized", str(LARGE_BP_ANNUALIZED),
    "--device", DEVICE,
    "--dtype", DTYPE,
]
print(" ".join(cmd))
run_stream(cmd, cwd=ROOT, env=env)


In [ ]:
# Inspect residual diagnostics and peak IRF summaries.
import json

for policy in ["fixed", "ba", "bottleneck", "repair_aware"]:
    eval_path = SHOCK_ROOT / f"{policy}_monetary_shock_eval.json"
    if eval_path.exists():
        print(policy, json.loads(eval_path.read_text())["overall.rms"])

summary_path = SHOCK_ROOT / "postprocess" / "monetary_shock_peak_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2)[:4000])


In [ ]:
# Plot deviations from the no-shock path for the larger monetary shock.
import matplotlib.pyplot as plt
import numpy as np

variables = ["R", "Pi", "chi", "I_A", "A", "output_gap"]
scenario = f"mp_{LARGE_BP_ANNUALIZED}bp"
plot_policies = ["fixed", "ba", "bottleneck", "repair_aware"]
fig, axes = plt.subplots(len(variables), len(plot_policies), figsize=(18, 12), sharex=True)

for col, policy in enumerate(plot_policies):
    path = SHOCK_ROOT / "postprocess" / f"IR_{policy}_monetary_shock_definitions.npz"
    data = np.load(path)
    for row, variable in enumerate(variables):
        base = data[f"no_shock__{variable}"]
        shocked = data[f"{scenario}__{variable}"]
        axes[row, col].plot(shocked - base)
        axes[row, col].axhline(0, color="black", linewidth=0.6)
        axes[row, col].set_title(f"{policy}: {variable}")

fig.tight_layout()
plt.show()
